# CPSA 03 — FP32 training on 2× T4

YOLOv8-pose with Hard-Swish, no attention, trained on the **mixed** dataset:
the *curated* half-bust crops from notebook 01 plus the full-frame images from
notebook 02, so the predicted box learns the true person extent while half-body
views stay dominant. Ultralytics trains square `imgsz=640` with `rect=False`;
the deployment graph is exported and audited at the camera's native 480×640
input.

The curated train list (see `cpsa_ml.datasets.curation`) keeps only crops at
least `MIN_HEIGHT` pixels tall and repeats the crops that show a hand
(`HAND_REPEAT`) or a face (`FACE_REPEAT`, 1 = off). `LABELS_ROOT` swaps in the
Task P pseudo-labels when that dataset is attached, and `EXTRA_DATA` mixes in
extra `halfbust62` datasets (HaGRID frames) at their own repeat factor.

Notebook 04 carries the tuned study recipe (`cos_lr`, explicit SGD,
`degrees=10`, `close_mosaic=15`, motion blur); this notebook keeps the
`cpsa_ml.train` defaults for the baseline Hard-Swish checkpoint.


In [ ]:
import subprocess
import sys

# Pin the exact CPSA_2026 commit for a reproducible Kaggle run. The coordinator
# pins the merge commit that contains this notebook and its cpsa_ml changes.
COMMIT_SHA = "a06184367d62061955867d6199b949383e659763"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"git+https://github.com/fcabecciaw/CPSA_2026@{COMMIT_SHA}#subdirectory=ml",
    ],
    check=True,
)

In [ ]:
from pathlib import Path

from cpsa_ml.datasets.curation import CurationConfig, curate_train_list
from cpsa_ml.datasets.staging import find_inputs, stage_dataset
from cpsa_ml.variant_study import build_mixed_data_yaml

# Kaggle mounts datasets under /kaggle/input/datasets/<owner>/<slug>/ (older
# kernels: /kaggle/input/<slug>/), so datasets are located by content.
INPUT = Path("/kaggle/input")
DATA = next(
    (
        p
        for p in find_inputs(INPUT, "data.yaml")
        if "cpsa-halfbust" in str(p) and "cpsa-halfbust-pl" not in str(p)
    ),
    None,
)  # the pseudo-label dataset is attached separately as LABELS_ROOT
if DATA is None:
    raise FileNotFoundError("attach fabiocabeccia/cpsa-halfbust (no data.yaml found)")
FULL_DATA = next(
    (p.parent for p in find_inputs(INPUT, "data.yaml") if "cpsa-fullframe" in str(p)), None
)  # notebook-02 dataset; None trains on the half-body set only

# Task P pseudo-labels: a dataset named cpsa-halfbust-pl with a labels/train
# tree; None trains on the ground-truth labels.
LABELS_ROOT = next(
    (
        p.parent.parent
        for p in find_inputs(INPUT, "train")
        if p.parent.name == "labels" and "cpsa-halfbust-pl" in str(p)
    ),
    None,
)

# Extra halfbust62 datasets mixed into the train list as (root, repeat) entries;
# empty when none is attached, so this notebook runs without them.
EXTRA_DATA = [(p.parent, 1) for p in find_inputs(INPUT, "data.yaml") if "cpsa-hagrid62" in str(p)]

MIN_HEIGHT = 128  # keep crops at least this tall (hand/face detail survives)
HAND_REPEAT = 2  # crops with a hand of >= 10 visible keypoints
FACE_REPEAT = 1  # crops with >= 5 visible face keypoints; 1 = off
SCHEMA = "halfbust62"

CURATION = curate_train_list(
    CurationConfig(
        root=DATA.parent,
        out_dir=Path("/kaggle/working"),
        min_height=MIN_HEIGHT,
        hand_repeat=HAND_REPEAT,
        face_repeat=FACE_REPEAT,
        labels_root=LABELS_ROOT,
        schema=SCHEMA,
        extra=EXTRA_DATA,
    )
)

MIXED_DATA = Path("/kaggle/working/data_mixed.yaml")
payload = build_mixed_data_yaml(
    DATA.parent,
    FULL_DATA,
    MIXED_DATA,
    schema=SCHEMA,
    curated_train=CURATION.train_list,
    val_h160=CURATION.val_h160_list,
)
# Copy the attached datasets to the local disk once: /kaggle/input is a slow
# network mount (about 100 files/s), which would stall the curation scan and
# the image loading of every epoch. The copy is reused if the cell is rerun.
LOCAL_COPY = True
if LOCAL_COPY:
    DATA = stage_dataset(DATA.parent, Path("/tmp/cpsa-halfbust")) / "data.yaml"
    if FULL_DATA is not None:
        FULL_DATA = stage_dataset(FULL_DATA, Path("/tmp/cpsa-fullframe"))
    EXTRA_DATA = [
        (stage_dataset(root, Path("/tmp") / f"extra_{i}"), repeat)
        for i, (root, repeat) in enumerate(EXTRA_DATA)
    ]
print("half-body data:", DATA, "| full-frame data:", FULL_DATA)
print("pseudo-labels:", LABELS_ROOT, "| extra datasets:", EXTRA_DATA)
print("train:", payload["train"])
print("val:", payload["val"], "| val_h160:", payload.get("val_h160"))

In [ ]:
from cpsa_ml.train import TrainConfig, train

VARIANT = "n"
EPOCHS = 100
BATCH = 32
DEVICE = "0,1"  # 2× T4 DDP
IMGSZ = 640
PROJECT = Path("/kaggle/working/runs")
NAME = "halfbust62-yolov8n-hswish-fp32"
RESUME = ""  # e.g. "/kaggle/input/cpsa-checkpoints/last.pt"

best = train(
    TrainConfig(
        data=MIXED_DATA,
        schema=SCHEMA,
        variant=VARIANT,
        epochs=EPOCHS,
        batch=BATCH,
        device=DEVICE,
        imgsz=IMGSZ,
        project=PROJECT,
        name=NAME,
        resume=Path(RESUME) if RESUME else None,
        seed=0,
        workers=4,
        amp=True,
        val=True,
        plots=True,
    )
)
print("best checkpoint:", best)

In [ ]:
from cpsa_ml.onnx_check import check_onnx, export_onnx, format_report

ONNX = Path("/kaggle/working/export/halfbust62.onnx")
onnx_path = export_onnx(best, ONNX, imgsz=(480, 640))
report = check_onnx(onnx_path, imgsz=(480, 640), opset=17)
print(format_report(report))
assert report.passed, "backbone/neck is not DPU-compatible"

In [ ]:
# Re-publish /kaggle/working/runs as a dataset version. In the next session set
# RESUME to the attached .../weights/last.pt.
for path in sorted((PROJECT / NAME / "weights").glob("*.pt")):
    print(path)